# Partial HuBERT fine-tuning for Mandarin tone recognition

This advanced experiment reproduces the architecture that reached
approximately **68% cross-speaker tone accuracy** in the pilot
study. It fine-tunes the top four Chinese HuBERT Transformer layers,
trains ordered eight-head temporal attention, and jointly predicts
the base syllable and tone. Boundary silence is regenerated every
training epoch.

Speaker 1 supplies training and validation data. Speaker 2 remains
completely external until evaluation. Speaker 3 is evaluated only
for exploratory base-syllable recognition because its prompts have
no tone labels.

Prompt labels are intended targets, not verified realizations. The
resulting predictions are not pronunciation grades.

## 1. Select an A100 runtime and install packages

Choose **Runtime → Change runtime type → A100 GPU**. This notebook
performs real encoder fine-tuning; a CPU runtime is not practical.

In [ ]:
%pip install -q -U                 "huggingface_hub>=0.34,<1"                 "transformers==4.55.2"                 "gradio>=5,<7"                 soundfile scipy pandas matplotlib

In [ ]:
import getpass
import json
import math
import random
import subprocess
import sys
import zipfile
from pathlib import Path

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download
from scipy.signal import resample_poly
from transformers import AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Select an A100 GPU runtime before continuing.")
print(torch.cuda.get_device_name())

## 2. Download the rate-limit-safe private release

Store a personal read token in Colab Secrets as `HF_TOKEN`. The
notebook downloads one versioned ZIP rather than thousands of Hub
objects.

In [ ]:
REPO_ID = "abecode/mandarin_isolated_syllables"
REVISION = "v0.1.1"
ARCHIVE_NAME = "mandarin_isolated_syllables_v0.1.1.zip"
DATASET_ROOT = Path("/content/mandarin_isolated_syllables")
WORK_DIR = Path("/content/mandarin_finetune")
WORK_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, KeyError, RuntimeError):
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face read token: ")

archive_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=ARCHIVE_NAME,
    revision=REVISION,
    token=HF_TOKEN,
)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    root = DATASET_ROOT.resolve()
    for member in archive.infolist():
        destination = (root / member.filename).resolve()
        if not destination.is_relative_to(root):
            raise ValueError(f"Unsafe archive path: {member.filename}")
    archive.extractall(DATASET_ROOT)
print("Extracted", len(list((DATASET_ROOT / "data/audio").rglob("*.flac"))), "recordings")

## 3. Install the tested experiment code in this runtime

The following cells contain the same project modules used by the
cluster experiment: endpoint detection, dynamic augmentation,
attention pooling, splitting, checkpoint handling, and partial
fine-tuning. Keeping these as modules also makes the saved
checkpoints reconstructable.

In [ ]:
%%writefile /content/mandarin_finetune/attention_pooling.py
"""Learned temporal pooling modules for utterance classification."""

from __future__ import annotations

import math

import torch
from torch import nn

ATTENTION_POOLING = {"attentive_global", "ordered8", "attentive_combined"}


def sequence_mask(lengths: torch.Tensor, width: int) -> torch.Tensor:
    """Return a Boolean mask for valid sequence positions."""
    positions = torch.arange(width, device=lengths.device).unsqueeze(0)
    return positions < lengths.unsqueeze(1)


class AttentiveStatisticsPooling(nn.Module):
    """Learn a scalar weight for every frame, then pool mean and deviation."""

    def __init__(self, width: int, attention_size: int = 128) -> None:
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(width, attention_size),
            nn.Tanh(),
            nn.Linear(attention_size, 1),
        )

    def forward(
        self, hidden: torch.Tensor, lengths: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        valid = sequence_mask(lengths, hidden.shape[1])
        logits = self.score(hidden).squeeze(-1).masked_fill(~valid, -torch.inf)
        attention = logits.softmax(dim=1)
        mean = torch.einsum("bt,btd->bd", attention, hidden)
        variance = torch.einsum(
            "bt,btd->bd", attention, (hidden - mean.unsqueeze(1)).square()
        )
        return torch.cat((mean, variance.clamp_min(1e-7).sqrt()), dim=1), attention


class OrderedAttentionPooling(nn.Module):
    """Pool a sequence with content-aware heads having learnable ordered windows."""

    def __init__(
        self,
        width: int,
        heads: int = 8,
        frame_projection_size: int = 128,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.heads = heads
        self.frame_projection_size = frame_projection_size
        attention_size = 128
        self.keys = nn.Sequential(nn.Linear(width, attention_size), nn.Tanh())
        self.queries = nn.Parameter(torch.empty(heads, attention_size))
        nn.init.normal_(self.queries, std=0.02)
        initial_centers = torch.linspace(0.08, 0.92, heads)
        self.center_logits = nn.Parameter(torch.logit(initial_centers))
        self.log_widths = nn.Parameter(torch.full((heads,), math.log(0.22)))
        self.frame_project = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, frame_projection_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(
        self, hidden: torch.Tensor, lengths: torch.Tensor
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        batch, frames, _ = hidden.shape
        valid = sequence_mask(lengths, frames)
        keys = self.keys(hidden)
        content = torch.einsum("btd,hd->bht", keys, self.queries) / math.sqrt(
            keys.shape[-1]
        )
        positions = torch.arange(frames, device=hidden.device).view(1, 1, -1)
        denominators = (lengths - 1).clamp_min(1).view(-1, 1, 1)
        relative_positions = positions / denominators
        centers = self.center_logits.sigmoid().view(1, -1, 1)
        widths = self.log_widths.exp().clamp(0.05, 1.0).view(1, -1, 1)
        position_bias = -0.5 * ((relative_positions - centers) / widths).square()
        logits = (content + position_bias).masked_fill(~valid.unsqueeze(1), -torch.inf)
        attention = logits.softmax(dim=2)
        projected = self.frame_project(hidden)
        summaries = torch.einsum("bht,btd->bhd", attention, projected)
        empirical_centers = (attention * relative_positions).sum(dim=2)
        normalized = attention / attention.square().sum(dim=2, keepdim=True).sqrt()
        similarity = torch.bmm(normalized, normalized.transpose(1, 2))
        identity = torch.eye(self.heads, device=hidden.device).unsqueeze(0)
        diversity_loss = (similarity - identity).square().mean()
        ordering_loss = torch.relu(
            empirical_centers[:, :-1] - empirical_centers[:, 1:] + 0.03
        ).mean()
        entropy = -(attention.clamp_min(1e-8).log() * attention).sum(dim=2)
        return summaries.flatten(1), {
            "attention": attention,
            "centers": empirical_centers,
            "entropy": entropy,
            "diversity_loss": diversity_loss,
            "ordering_loss": ordering_loss,
        }


In [ ]:
%%writefile /content/mandarin_finetune/checkpoint_utils.py
"""Versioned checkpoint helpers shared by training and inference code."""

from __future__ import annotations

from pathlib import Path
from typing import Any

import torch

CURRENT_CHECKPOINT_FORMAT = 1
SUPPORTED_FORMATS = {0, CURRENT_CHECKPOINT_FORMAT}
LEGACY_STATE_KEYS = {"state_dict", "trainable_state_dict"}
METRIC_KEYS = {"history", "validation", "external", "oli"}
MODEL_REVISIONS = {
    "TencentGameMate/chinese-hubert-base": "fce0375452b1dd6c080ac3248d423d4d037bc831",
    "facebook/wav2vec2-xls-r-300m": "1a640f32ac3e39899438a2931f9924c02f080a54",
}


def split_run_record(
    run_record: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Separate reconstruction metadata from measured results."""
    metadata = {
        key: value for key, value in run_record.items() if key not in METRIC_KEYS
    }
    metrics = {key: run_record[key] for key in METRIC_KEYS if key in run_record}
    return metadata, metrics


def create_checkpoint(
    *,
    state_dict: dict[str, torch.Tensor],
    metadata: dict[str, Any],
    metrics: dict[str, Any],
) -> dict[str, Any]:
    """Create a checkpoint using the current schema."""
    checkpoint = {
        "format": CURRENT_CHECKPOINT_FORMAT,
        "state_dict": state_dict,
        "metadata": metadata,
        "metrics": metrics,
    }
    validate_checkpoint(checkpoint)
    return checkpoint


def validate_checkpoint(checkpoint: dict[str, Any]) -> None:
    """Validate a checkpoint without changing its format."""
    checkpoint_format = checkpoint.get("format", 0)
    if checkpoint_format not in SUPPORTED_FORMATS:
        raise ValueError(f"Unsupported checkpoint format: {checkpoint_format!r}")

    if checkpoint_format == 0:
        state_keys = LEGACY_STATE_KEYS.intersection(checkpoint)
        if len(state_keys) != 1:
            raise ValueError(
                "Format-0 checkpoint must contain exactly one state dictionary; "
                f"found {sorted(state_keys)}"
            )
        if not isinstance(checkpoint.get("metrics"), dict):
            raise TypeError("Format-0 checkpoint 'metrics' must be a dictionary")
        return

    expected_keys = {"format", "state_dict", "metadata", "metrics"}
    if set(checkpoint) != expected_keys:
        raise ValueError(
            "Format-1 checkpoint keys must be exactly "
            f"{sorted(expected_keys)}; found {sorted(checkpoint)}"
        )
    if not isinstance(checkpoint["state_dict"], dict):
        raise TypeError("Checkpoint 'state_dict' must be a dictionary")
    if not isinstance(checkpoint["metadata"], dict):
        raise TypeError("Checkpoint 'metadata' must be a dictionary")
    if not isinstance(checkpoint["metrics"], dict):
        raise TypeError("Checkpoint 'metrics' must be a dictionary")

    metadata = checkpoint["metadata"]
    required_metadata = {
        "checkpoint_kind",
        "state_scope",
        "model_name",
        "model_revision",
        "pooling",
        "base_vocabulary",
        "architecture",
    }
    missing = required_metadata.difference(metadata)
    if missing:
        raise ValueError(f"Checkpoint metadata is missing: {sorted(missing)}")
    if metadata["state_scope"] not in {"complete_head", "trainable_overlay"}:
        raise ValueError(f"Unknown state scope: {metadata['state_scope']!r}")


def load_raw_checkpoint(path: Path) -> dict[str, Any]:
    """Load a trusted project checkpoint onto CPU without conversion."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint must be a dictionary")
    validate_checkpoint(checkpoint)
    return checkpoint


def convert_to_current_format(checkpoint: dict[str, Any]) -> dict[str, Any]:
    """Return a format-1 representation of a supported checkpoint."""
    validate_checkpoint(checkpoint)
    if checkpoint.get("format", 0) == CURRENT_CHECKPOINT_FORMAT:
        return checkpoint

    combined_record = checkpoint["metrics"]
    metadata, metrics = split_run_record(combined_record)
    is_overlay = "trainable_state_dict" in checkpoint
    model_name = metadata["model_name"]
    pooling = metadata["pooling"]
    metadata.update(
        {
            "checkpoint_kind": (
                "partial_finetune" if is_overlay else "frozen_encoder_classifier"
            ),
            "state_scope": "trainable_overlay" if is_overlay else "complete_head",
            "model_revision": MODEL_REVISIONS[model_name],
            "architecture": {
                "dropout": 0.2,
                "projection_size": 256,
                "temporal_bins": 8 if pooling == "temporal8" else None,
                "frame_projection_size": 128 if pooling == "temporal8" else None,
            },
        }
    )
    legacy_state_key = "trainable_state_dict" if is_overlay else "state_dict"
    return create_checkpoint(
        state_dict=checkpoint[legacy_state_key],
        metadata=metadata,
        metrics=metrics,
    )


def load_checkpoint(path: Path) -> dict[str, Any]:
    """Load and normalize a trusted project checkpoint to the current format."""
    return convert_to_current_format(load_raw_checkpoint(path))


def save_checkpoint(path: Path, checkpoint: dict[str, Any]) -> None:
    """Validate and atomically save a checkpoint.

    The temporary file is written beside the destination so replacement is
    atomic on the project filesystem. An interrupted write leaves the previous
    checkpoint intact.
    """
    validate_checkpoint(checkpoint)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp")
    try:
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)


In [ ]:
%%writefile /content/mandarin_finetune/config_utils.py
"""Small JSON configuration helpers for command-line experiments."""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path
from typing import Any


def requested_config_path(default: Path) -> Path:
    """Read only --config from argv before constructing the complete parser."""
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument("--config", type=Path, default=default)
    known, _ = parser.parse_known_args(sys.argv[1:])
    return known.config


def apply_config_defaults(
    parser: argparse.ArgumentParser, path: Path
) -> dict[str, Any]:
    """Load JSON values and use them as argparse defaults."""
    values = json.loads(path.read_text(encoding="utf-8"))
    valid_destinations = {action.dest for action in parser._actions}
    unknown = set(values).difference(valid_destinations)
    if unknown:
        raise ValueError(f"Unknown configuration keys in {path}: {sorted(unknown)}")
    parser.set_defaults(**values)
    return values


In [ ]:
%%writefile /content/mandarin_finetune/extract_speech_features.py
#!/usr/bin/env python3
"""Extract frozen speech-encoder features for all experiment recordings."""

from __future__ import annotations

import argparse
import csv
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from checkpoint_utils import MODEL_REVISIONS
from transformers import AutoModel

MODEL_NAMES = {
    "hubert": "TencentGameMate/chinese-hubert-base",
    "xlsr": "facebook/wav2vec2-xls-r-300m",
}


def decode_audio(path: Path, ffmpeg: Path) -> torch.Tensor:
    command = [
        str(ffmpeg),
        "-v",
        "error",
        "-i",
        str(path),
        "-ac",
        "1",
        "-ar",
        "16000",
        "-f",
        "f32le",
        "-",
    ]
    result = subprocess.run(command, check=True, stdout=subprocess.PIPE)
    audio = np.frombuffer(result.stdout, dtype=np.float32).copy()
    if audio.size == 0:
        raise ValueError(f"Decoded empty audio: {path}")
    return torch.from_numpy(audio)


def collate_audio(
    items: list[tuple[dict[str, str], torch.Tensor]],
) -> tuple[torch.Tensor, torch.Tensor]:
    length = max(audio.numel() for _, audio in items)
    values = torch.zeros(len(items), length, dtype=torch.float32)
    mask = torch.zeros(len(items), length, dtype=torch.long)
    for index, (_, audio) in enumerate(items):
        values[index, : audio.numel()] = audio
        mask[index, : audio.numel()] = 1
    return values, mask


def pool_hidden(
    hidden: torch.Tensor, output_lengths: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    global_features = []
    temporal_features = []
    for sequence, length in zip(hidden, output_lengths.tolist()):
        sequence = sequence[: max(1, int(length))].float()
        mean = sequence.mean(dim=0)
        std = sequence.std(dim=0, unbiased=False)
        global_features.append(torch.cat((mean, std)))
        # Adaptive pooling partitions relative syllable time into eight ordered regions.
        temporal = F.adaptive_avg_pool1d(sequence.T.unsqueeze(0), 8).squeeze(0).T
        temporal_features.append(temporal)
    return torch.stack(global_features), torch.stack(temporal_features)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--encoder", choices=sorted(MODEL_NAMES), required=True)
    parser.add_argument("--manifest", type=Path, default=Path("data/recordings.csv"))
    parser.add_argument("--output", type=Path)
    parser.add_argument("--model-cache", type=Path, default=Path("models/huggingface"))
    parser.add_argument("--ffmpeg", type=Path, default=Path("models/linux/ffmpeg"))
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--limit", type=int)
    args = parser.parse_args()

    output = args.output or Path(f"data/features/{args.encoder}.pt")
    with args.manifest.open(newline="", encoding="utf-8") as handle:
        rows = [
            row for row in csv.DictReader(handle) if row["include_experiment"] == "yes"
        ]
    if args.limit is not None:
        rows = rows[: args.limit]

    device = torch.device(args.device)
    model_name = MODEL_NAMES[args.encoder]
    model_revision = MODEL_REVISIONS[model_name]
    model = (
        AutoModel.from_pretrained(
            model_name,
            revision=model_revision,
            cache_dir=args.model_cache,
        )
        .to(device)
        .eval()
    )
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    paths: list[str] = []
    bases: list[str] = []
    tones: list[str] = []
    datasets: list[str] = []
    global_batches: list[torch.Tensor] = []
    temporal_batches: list[torch.Tensor] = []

    for start in range(0, len(rows), args.batch_size):
        batch_rows = rows[start : start + args.batch_size]
        decoded = [
            (row, decode_audio(Path(row["path"]), args.ffmpeg)) for row in batch_rows
        ]
        values, input_mask = collate_audio(decoded)
        with (
            torch.inference_mode(),
            torch.autocast(device_type=device.type, enabled=device.type == "cuda"),
        ):
            hidden = model(
                input_values=values.to(device), attention_mask=input_mask.to(device)
            ).last_hidden_state
        if hasattr(model, "_get_feat_extract_output_lengths"):
            lengths = model._get_feat_extract_output_lengths(
                input_mask.sum(dim=1)
            ).cpu()
        else:
            lengths = torch.full((len(batch_rows),), hidden.shape[1], dtype=torch.long)
        global_features, temporal_features = pool_hidden(hidden.cpu(), lengths)
        global_batches.append(global_features.half())
        temporal_batches.append(temporal_features.half())
        paths.extend(row["path"] for row in batch_rows)
        bases.extend(row["canonical_base"] for row in batch_rows)
        tones.extend(row["canonical_tone"] for row in batch_rows)
        datasets.extend(row["dataset"] for row in batch_rows)
        print(f"{min(start + args.batch_size, len(rows))}/{len(rows)}", flush=True)

    artifact = {
        "encoder": args.encoder,
        "model_name": model_name,
        "model_revision": model_revision,
        "paths": paths,
        "bases": bases,
        "tones": tones,
        "datasets": datasets,
        "global": torch.cat(global_batches),
        "temporal8": torch.cat(temporal_batches),
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(artifact, output)
    print(f"Wrote {len(paths)} recordings to {output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mandarin_finetune/silence_augmentation.py
"""Boundary-silence augmentation for untrimmed isolated speech."""

from __future__ import annotations

import random
from dataclasses import dataclass

import torch


@dataclass(frozen=True)
class SilenceAugmentationConfig:
    """Parameters for randomized leading and trailing nonspeech."""

    sample_rate: int = 16_000
    maximum_ms: int = 500
    zero_probability: float = 0.25
    noise_probability: float = 0.25


def _repeat_to_length(source: torch.Tensor, length: int) -> torch.Tensor:
    if length == 0:
        return source.new_empty(0)
    if source.numel() == 0:
        return source.new_zeros(length)
    repeats = (length + source.numel() - 1) // source.numel()
    return source.repeat(repeats)[:length]


def _boundary_audio(
    waveform: torch.Tensor,
    metadata: dict,
    length: int,
    leading: bool,
    config: SilenceAugmentationConfig,
) -> torch.Tensor:
    draw = random.random()
    if draw < config.zero_probability:
        return waveform.new_zeros(length)
    if leading:
        source = waveform[: int(metadata["start_sample"])]
    else:
        source = waveform[int(metadata["end_sample"]) :]
    if draw < config.zero_probability + config.noise_probability or not source.numel():
        boundary = waveform[: min(waveform.numel(), config.sample_rate // 10)]
        rms = boundary.square().mean().sqrt().clamp_min(1e-5)
        return torch.randn(length, dtype=waveform.dtype) * rms
    if source.numel() > length:
        start = random.randint(0, source.numel() - length)
        return source[start : start + length]
    return _repeat_to_length(source, length)


def add_boundary_silence(
    waveform: torch.Tensor,
    metadata: dict,
    config: SilenceAugmentationConfig = SilenceAugmentationConfig(),
) -> tuple[torch.Tensor, tuple[int, int]]:
    """Add independently sampled valid nonspeech to both waveform boundaries."""
    maximum = round(config.sample_rate * config.maximum_ms / 1_000)
    leading_length = random.randint(0, maximum)
    trailing_length = random.randint(0, maximum)
    leading = _boundary_audio(waveform, metadata, leading_length, True, config)
    trailing = _boundary_audio(waveform, metadata, trailing_length, False, config)
    return torch.cat((leading, waveform, trailing)), (leading_length, trailing_length)


In [ ]:
%%writefile /content/mandarin_finetune/speech_endpointing.py
#!/usr/bin/env python3
"""Conservative energy-based endpoint detection for isolated syllables."""

from __future__ import annotations

from dataclasses import asdict, dataclass

import torch


@dataclass(frozen=True)
class EndpointConfig:
    """Parameters controlling frame-energy endpoint detection."""

    sample_rate: int = 16_000
    frame_ms: float = 20.0
    hop_ms: float = 10.0
    noise_percentile: float = 0.2
    noise_margin_db: float = 8.0
    peak_floor_db: float = -35.0
    minimum_active_ms: float = 40.0
    maximum_gap_ms: float = 100.0
    margin_ms: float = 80.0
    minimum_retained_ms: float = 250.0
    maximum_trim_fraction: float = 0.8

    def to_dict(self) -> dict[str, float | int]:
        """Return a serialization-friendly representation."""
        return asdict(self)


@dataclass(frozen=True)
class EndpointResult:
    """Detected speech span and diagnostic frame energies."""

    start_sample: int
    end_sample: int
    threshold_db: float
    noise_floor_db: float
    peak_db: float
    fallback: bool
    fallback_reason: str
    frame_energy_db: torch.Tensor
    active_frames: torch.Tensor


def _runs(values: torch.Tensor) -> list[tuple[int, int, bool]]:
    """Return half-open runs from a one-dimensional Boolean tensor."""
    if values.numel() == 0:
        return []
    result = []
    start = 0
    current = bool(values[0])
    for index in range(1, values.numel()):
        value = bool(values[index])
        if value != current:
            result.append((start, index, current))
            start = index
            current = value
    result.append((start, values.numel(), current))
    return result


def _clean_activity(
    active: torch.Tensor, minimum_frames: int, maximum_gap_frames: int
) -> torch.Tensor:
    """Remove brief active runs and bridge brief internal gaps."""
    cleaned = active.clone()
    for start, end, value in _runs(cleaned):
        if value and end - start < minimum_frames:
            cleaned[start:end] = False
    runs = _runs(cleaned)
    for run_index, (start, end, value) in enumerate(runs):
        internal = 0 < run_index < len(runs) - 1
        if not value and internal and end - start <= maximum_gap_frames:
            cleaned[start:end] = True
    return cleaned


def detect_endpoints(
    waveform: torch.Tensor, config: EndpointConfig = EndpointConfig()
) -> EndpointResult:
    """Find a conservative speech span using recording-adaptive RMS energy."""
    audio = waveform.detach().float().flatten()
    frame_length = round(config.sample_rate * config.frame_ms / 1_000)
    hop_length = round(config.sample_rate * config.hop_ms / 1_000)
    if audio.numel() < frame_length:
        return _fallback(audio, "shorter_than_one_frame")

    frames = audio.unfold(0, frame_length, hop_length)
    rms = frames.square().mean(dim=1).sqrt().clamp_min(1e-8)
    energy_db = 20 * torch.log10(rms)
    sorted_energy = energy_db.sort().values
    noise_index = min(
        sorted_energy.numel() - 1,
        max(0, round(config.noise_percentile * (sorted_energy.numel() - 1))),
    )
    noise_floor_db = float(sorted_energy[noise_index])
    peak_db = float(energy_db.max())
    threshold_db = max(
        noise_floor_db + config.noise_margin_db,
        peak_db + config.peak_floor_db,
    )
    active = energy_db >= threshold_db
    minimum_frames = max(1, round(config.minimum_active_ms / config.hop_ms))
    maximum_gap_frames = max(0, round(config.maximum_gap_ms / config.hop_ms))
    active = _clean_activity(active, minimum_frames, maximum_gap_frames)
    indices = active.nonzero().flatten()
    if indices.numel() == 0:
        return _fallback(
            audio,
            "no_active_frames",
            energy_db,
            active,
            threshold_db,
            noise_floor_db,
            peak_db,
        )

    margin = round(config.sample_rate * config.margin_ms / 1_000)
    start = max(0, int(indices[0]) * hop_length - margin)
    end = min(audio.numel(), int(indices[-1]) * hop_length + frame_length + margin)
    minimum_samples = round(config.sample_rate * config.minimum_retained_ms / 1_000)
    trim_fraction = 1 - (end - start) / audio.numel()
    if end - start < minimum_samples:
        reason = "retained_span_too_short"
    elif trim_fraction > config.maximum_trim_fraction:
        reason = "trim_fraction_too_large"
    else:
        reason = ""
    if reason:
        return _fallback(
            audio,
            reason,
            energy_db,
            active,
            threshold_db,
            noise_floor_db,
            peak_db,
        )
    return EndpointResult(
        start,
        end,
        threshold_db,
        noise_floor_db,
        peak_db,
        False,
        "",
        energy_db,
        active,
    )


def _fallback(
    audio: torch.Tensor,
    reason: str,
    energy_db: torch.Tensor | None = None,
    active: torch.Tensor | None = None,
    threshold_db: float = float("nan"),
    noise_floor_db: float = float("nan"),
    peak_db: float = float("nan"),
) -> EndpointResult:
    """Construct a result retaining the original waveform."""
    if energy_db is None:
        energy_db = torch.empty(0)
    if active is None:
        active = torch.empty(0, dtype=torch.bool)
    return EndpointResult(
        0,
        audio.numel(),
        threshold_db,
        noise_floor_db,
        peak_db,
        True,
        reason,
        energy_db,
        active,
    )


In [ ]:
%%writefile /content/mandarin_finetune/train_syllable_classifier.py
#!/usr/bin/env python3
"""Train base-syllable and tone heads on cached frozen-encoder features."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from checkpoint_utils import (
    MODEL_REVISIONS,
    create_checkpoint,
    save_checkpoint,
    split_run_record,
)
from config_utils import apply_config_defaults, requested_config_path
from torch import nn
from torch.utils.data import DataLoader, Dataset

ABE_DATASETS = {"tone_labeled", "abe_new"}
YUE_DATASETS = {"tone_labeled_yue"}
OLI_DATASETS = {"tone_unspecified"}


def speaker_group(dataset: str) -> str:
    if dataset in ABE_DATASETS:
        return "abe"
    if dataset in YUE_DATASETS:
        return "yue"
    if dataset in OLI_DATASETS:
        return "oli"
    raise ValueError(f"Unknown dataset: {dataset}")


def validation_member(path: str, fraction: float, seed: int) -> bool:
    digest = hashlib.sha256(f"{seed}:{path}".encode()).digest()
    value = int.from_bytes(digest[:8], "big") / 2**64
    return value < fraction


def stable_order(path: str, seed: int) -> bytes:
    return hashlib.sha256(f"{seed}:{path}".encode()).digest()


def split_training_speaker(
    paths: list[str],
    bases: list[str],
    groups: list[str],
    train_speaker: str,
    strategy: str,
    fraction: float,
    seed: int,
) -> tuple[list[int], list[int]]:
    candidates = [index for index, group in enumerate(groups) if group == train_speaker]
    if strategy == "hash-fraction":
        validation = [
            index
            for index in candidates
            if validation_member(paths[index], fraction, seed)
        ]
        validation_set = set(validation)
        return [
            index for index in candidates if index not in validation_set
        ], validation

    by_base: dict[str, list[int]] = {}
    for index in candidates:
        by_base.setdefault(bases[index], []).append(index)
    validation = []
    for indices in by_base.values():
        # Never remove the only training example of a class.
        if len(indices) >= 2:
            validation.append(
                min(indices, key=lambda index: stable_order(paths[index], seed))
            )
    validation_set = set(validation)
    return [index for index in candidates if index not in validation_set], sorted(
        validation
    )


class FeatureDataset(Dataset):
    def __init__(
        self,
        features: torch.Tensor,
        base: torch.Tensor,
        tone: torch.Tensor,
        indices: list[int],
    ):
        self.features = features
        self.base = base
        self.tone = tone
        self.indices = indices

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        row = self.indices[index]
        return self.features[row].float(), self.base[row], self.tone[row], row


class Classifier(nn.Module):
    def __init__(
        self, shape: tuple[int, ...], pooling: str, bases: int, dropout: float
    ):
        super().__init__()
        self.pooling = pooling
        if pooling == "global":
            width = shape[0]
            self.project = nn.Sequential(
                nn.LayerNorm(width),
                nn.Linear(width, 256),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            width = shape[1]
            self.frame_project = nn.Sequential(
                nn.LayerNorm(width),
                nn.Linear(width, 128),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.project = nn.Sequential(
                nn.LayerNorm(8 * 128),
                nn.Linear(8 * 128, 256),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        self.base_head = nn.Linear(256, bases)
        self.tone_head = nn.Linear(256, 4)

    def forward(self, features: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if self.pooling == "temporal8":
            features = self.frame_project(features).flatten(1)
        shared = self.project(features)
        return self.base_head(shared), self.tone_head(shared)


def score(
    model: nn.Module, loader: DataLoader, device: torch.device
) -> tuple[dict, list[dict]]:
    model.eval()
    base_correct = tone_correct = joint_correct = tone_count = 0
    count = 0
    predictions = []
    with torch.inference_mode():
        for features, base, tone, rows in loader:
            base_logits, tone_logits = model(features.to(device))
            base_pred = base_logits.argmax(1).cpu()
            tone_pred = tone_logits.argmax(1).cpu()
            base_correct += (base_pred == base).sum().item()
            valid_tone = tone >= 0
            tone_correct += ((tone_pred == tone) & valid_tone).sum().item()
            joint_correct += (
                ((base_pred == base) & (tone_pred == tone) & valid_tone).sum().item()
            )
            tone_count += valid_tone.sum().item()
            count += base.numel()
            predictions.extend(
                {
                    "row": int(row),
                    "base_true": int(b),
                    "base_pred": int(bp),
                    "tone_true": int(t),
                    "tone_pred": int(tp),
                }
                for row, b, bp, t, tp in zip(rows, base, base_pred, tone, tone_pred)
            )
    metrics = {
        "n": count,
        "base_accuracy": base_correct / count if count else None,
        "tone_n": tone_count,
        "tone_accuracy": tone_correct / tone_count if tone_count else None,
        "joint_accuracy": joint_correct / tone_count if tone_count else None,
    }
    return metrics, predictions


def main() -> None:
    parser = argparse.ArgumentParser()
    default_config = Path("configs/frozen_classifier.json")
    parser.add_argument("--config", type=Path, default=default_config)
    parser.add_argument("--features", type=Path, required=True)
    parser.add_argument("--train-speaker", choices=["abe", "yue"], required=True)
    parser.add_argument("--pooling", choices=["global", "temporal8"], required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--patience", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--tone-loss-weight", type=float, default=1.0)
    parser.add_argument("--validation-fraction", type=float, default=0.15)
    parser.add_argument(
        "--validation-strategy",
        choices=["stratified-base", "hash-fraction"],
        default="stratified-base",
    )
    parser.add_argument("--dropout", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=20260821)
    parser.add_argument("--device", default="cuda")
    apply_config_defaults(parser, requested_config_path(default_config))
    args = parser.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    artifact = torch.load(args.features, map_location="cpu", weights_only=False)
    features = artifact[args.pooling]
    groups = [speaker_group(dataset) for dataset in artifact["datasets"]]
    base_names = sorted(set(artifact["bases"]))
    base_to_id = {name: index for index, name in enumerate(base_names)}
    base_targets = torch.tensor([base_to_id[name] for name in artifact["bases"]])
    # Neutral tone 5 and unspecified tones are masked from the symmetric 1--4 tone task.
    tone_targets = torch.tensor(
        [
            int(tone) - 1 if tone in {"1", "2", "3", "4"} else -1
            for tone in artifact["tones"]
        ]
    )

    train_indices, validation_indices = split_training_speaker(
        artifact["paths"],
        artifact["bases"],
        groups,
        args.train_speaker,
        args.validation_strategy,
        args.validation_fraction,
        args.seed,
    )
    external_indices, oli_indices = [], []
    external_speaker = "yue" if args.train_speaker == "abe" else "abe"
    for index, group in enumerate(groups):
        if group == external_speaker:
            external_indices.append(index)
        elif group == "oli":
            oli_indices.append(index)

    datasets = {
        "train": FeatureDataset(features, base_targets, tone_targets, train_indices),
        "validation": FeatureDataset(
            features, base_targets, tone_targets, validation_indices
        ),
        "external": FeatureDataset(
            features, base_targets, tone_targets, external_indices
        ),
        "oli": FeatureDataset(features, base_targets, tone_targets, oli_indices),
    }
    loaders = {
        name: DataLoader(
            data, batch_size=args.batch_size, shuffle=name == "train", num_workers=0
        )
        for name, data in datasets.items()
    }
    device = torch.device(
        args.device if args.device != "cuda" or torch.cuda.is_available() else "cpu"
    )
    model = Classifier(
        tuple(features.shape[1:]), args.pooling, len(base_names), args.dropout
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=args.learning_rate, weight_decay=1e-2
    )
    base_loss_fn = nn.CrossEntropyLoss()
    tone_loss_fn = nn.CrossEntropyLoss()

    best_validation = -1.0
    best_state = None
    stale = 0
    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        for batch_features, base, tone, _ in loaders["train"]:
            batch_features, base, tone = (
                batch_features.to(device),
                base.to(device),
                tone.to(device),
            )
            optimizer.zero_grad(set_to_none=True)
            base_logits, tone_logits = model(batch_features)
            loss = base_loss_fn(base_logits, base)
            valid_tone = tone >= 0
            if valid_tone.any():
                loss = loss + args.tone_loss_weight * tone_loss_fn(
                    tone_logits[valid_tone], tone[valid_tone]
                )
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * base.numel()
        validation, _ = score(model, loaders["validation"], device)
        selection = validation["base_accuracy"] + (validation["tone_accuracy"] or 0.0)
        record = {
            "epoch": epoch,
            "train_loss": total_loss / len(datasets["train"]),
            **validation,
        }
        history.append(record)
        print(json.dumps(record), flush=True)
        if selection > best_validation:
            best_validation = selection
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1
            if stale >= args.patience:
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    model_revision = artifact.get("model_revision") or MODEL_REVISIONS.get(
        artifact["model_name"]
    )
    if model_revision is None:
        raise ValueError(f"No pinned revision is known for {artifact['model_name']!r}")
    metrics = {
        "checkpoint_kind": "frozen_encoder_classifier",
        "state_scope": "complete_head",
        "encoder": artifact["encoder"],
        "model_name": artifact["model_name"],
        "model_revision": model_revision,
        "train_speaker": args.train_speaker,
        "external_speaker": external_speaker,
        "pooling": args.pooling,
        "seed": args.seed,
        "validation_strategy": args.validation_strategy,
        "base_vocabulary_size": len(base_names),
        "base_vocabulary": base_names,
        "architecture": {
            "dropout": args.dropout,
            "projection_size": 256,
            "temporal_bins": 8 if args.pooling == "temporal8" else None,
            "frame_projection_size": 128 if args.pooling == "temporal8" else None,
        },
        "training": {
            "epochs": args.epochs,
            "patience": args.patience,
            "batch_size": args.batch_size,
            "learning_rate": args.learning_rate,
            "tone_loss_weight": args.tone_loss_weight,
            "validation_fraction": args.validation_fraction,
        },
        "split_sizes": {name: len(data) for name, data in datasets.items()},
        "history": history,
    }
    all_predictions = {}
    for name in ("validation", "external", "oli"):
        metrics[name], all_predictions[name] = score(model, loaders[name], device)
    # Tone metrics for Oli are intentionally null because its labels are unspecified.
    (args.output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    metadata, measured_metrics = split_run_record(metrics)
    checkpoint = create_checkpoint(
        state_dict=best_state,
        metadata=metadata,
        metrics=measured_metrics,
    )
    save_checkpoint(args.output_dir / "classifier.pt", checkpoint)
    with (args.output_dir / "predictions.tsv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        fields = ["split", "path", "base_true", "base_pred", "tone_true", "tone_pred"]
        writer = csv.DictWriter(handle, fieldnames=fields, delimiter="\t")
        writer.writeheader()
        for split, predictions in all_predictions.items():
            for prediction in predictions:
                row = prediction.pop("row")
                tone_true = (
                    prediction["tone_true"] + 1 if prediction["tone_true"] >= 0 else ""
                )
                writer.writerow(
                    {
                        "split": split,
                        "path": artifact["paths"][row],
                        "base_true": base_names[prediction["base_true"]],
                        "base_pred": base_names[prediction["base_pred"]],
                        "tone_true": tone_true,
                        "tone_pred": prediction["tone_pred"] + 1,
                    }
                )
    print(
        json.dumps(
            {key: metrics[key] for key in ("validation", "external", "oli")}, indent=2
        )
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mandarin_finetune/train_unfrozen_classifier.py
#!/usr/bin/env python3
"""Partially fine-tune a speech encoder with syllable and tone heads."""

from __future__ import annotations

import argparse
import copy
import csv
import json
import random
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from attention_pooling import (
    ATTENTION_POOLING,
    AttentiveStatisticsPooling,
    OrderedAttentionPooling,
)
from checkpoint_utils import (
    MODEL_REVISIONS,
    create_checkpoint,
    save_checkpoint,
    split_run_record,
)
from config_utils import apply_config_defaults, requested_config_path
from extract_speech_features import MODEL_NAMES
from silence_augmentation import (
    SilenceAugmentationConfig,
    add_boundary_silence,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from train_syllable_classifier import speaker_group, split_training_speaker
from transformers import AutoModel

TEMPORAL_POOLING = {
    "temporal8": {"bins": 8, "frame_projection_size": 128},
    "temporal16": {"bins": 16, "frame_projection_size": 64},
}


class AudioDataset(Dataset):
    def __init__(
        self, artifact: dict, base: torch.Tensor, tone: torch.Tensor, indices: list[int]
    ):
        self.artifact = artifact
        self.base = base
        self.tone = tone
        self.indices = indices

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        row = self.indices[index]
        return self.artifact["waveforms"][row], self.base[row], self.tone[row], row


def collate_waveforms(items):
    length = max(item[0].numel() for item in items)
    values = torch.zeros(len(items), length)
    mask = torch.zeros(len(items), length, dtype=torch.long)
    base = torch.empty(len(items), dtype=torch.long)
    tone = torch.empty(len(items), dtype=torch.long)
    rows = torch.empty(len(items), dtype=torch.long)
    for index, (audio, base_target, tone_target, row) in enumerate(items):
        values[index, : audio.numel()] = audio
        mask[index, : audio.numel()] = 1
        base[index], tone[index], rows[index] = base_target, tone_target, row
    return values, mask, base, tone, rows


class AudioCollator:
    """Optionally add boundary nonspeech before padding an audio batch."""

    def __init__(
        self,
        augment: bool = False,
        endpoint_metadata: list[dict] | None = None,
        augmentation_config: SilenceAugmentationConfig = SilenceAugmentationConfig(),
    ) -> None:
        self.augment = augment
        self.endpoint_metadata = endpoint_metadata
        self.augmentation_config = augmentation_config

    def __call__(self, items):
        if not self.augment:
            return collate_waveforms(items)
        if self.endpoint_metadata is None:
            raise ValueError("Silence augmentation requires endpoint metadata")
        augmented = []
        for waveform, base, tone, row in items:
            waveform, _ = add_boundary_silence(
                waveform,
                self.endpoint_metadata[row],
                self.augmentation_config,
            )
            augmented.append((waveform, base, tone, row))
        return collate_waveforms(augmented)


class FineTuneModel(nn.Module):
    def __init__(self, encoder: nn.Module, pooling: str, bases: int, dropout: float):
        super().__init__()
        self.encoder = encoder
        self.pooling = pooling
        width = encoder.config.hidden_size
        self.attentive_global = None
        self.ordered_attention = None
        if pooling == "global":
            self.temporal_bins = None
            self.frame_projection_size = None
            self.project = nn.Sequential(
                nn.LayerNorm(width * 2),
                nn.Linear(width * 2, 256),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        elif pooling in TEMPORAL_POOLING:
            temporal_config = TEMPORAL_POOLING[pooling]
            self.temporal_bins = temporal_config["bins"]
            self.frame_projection_size = temporal_config["frame_projection_size"]
            self.frame_project = nn.Sequential(
                nn.LayerNorm(width),
                nn.Linear(width, self.frame_projection_size),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.project = nn.Sequential(
                nn.LayerNorm(self.temporal_bins * self.frame_projection_size),
                nn.Linear(self.temporal_bins * self.frame_projection_size, 256),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        elif pooling == "attentive_global":
            self.temporal_bins = None
            self.frame_projection_size = None
            self.attentive_global = AttentiveStatisticsPooling(width)
            self.project = self._projector(width * 2, 256, dropout)
        elif pooling == "ordered8":
            self.temporal_bins = 8
            self.frame_projection_size = 128
            self.ordered_attention = OrderedAttentionPooling(width, dropout=dropout)
            self.project = self._projector(8 * 128, 256, dropout)
        elif pooling == "attentive_combined":
            self.temporal_bins = 8
            self.frame_projection_size = 128
            self.attentive_global = AttentiveStatisticsPooling(width)
            self.ordered_attention = OrderedAttentionPooling(width, dropout=dropout)
            self.global_project = self._projector(width * 2, 128, dropout)
            self.ordered_project = self._projector(8 * 128, 128, dropout)
            self.project = self._projector(256, 256, dropout)
        else:
            raise ValueError(f"Unknown pooling: {pooling}")
        self.base_head = nn.Linear(256, bases)
        self.tone_head = nn.Linear(256, 4)

    @staticmethod
    def _projector(input_size: int, output_size: int, dropout: float) -> nn.Sequential:
        return nn.Sequential(
            nn.LayerNorm(input_size),
            nn.Linear(input_size, output_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def aggregate(
        self, hidden: torch.Tensor, lengths: torch.Tensor
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        if self.pooling in TEMPORAL_POOLING:
            pooled = []
            for sequence, length in zip(hidden, lengths.tolist()):
                sequence = self.frame_project(sequence[: max(1, int(length))])
                pooled.append(
                    F.adaptive_avg_pool1d(
                        sequence.T.unsqueeze(0), self.temporal_bins
                    ).flatten()
                )
            return torch.stack(pooled), {}
        if self.pooling in ATTENTION_POOLING:
            auxiliary = {}
            if self.attentive_global is not None:
                global_summary, global_attention = self.attentive_global(
                    hidden, lengths
                )
                auxiliary["global_attention"] = global_attention
            if self.ordered_attention is not None:
                ordered_summary, ordered_auxiliary = self.ordered_attention(
                    hidden, lengths
                )
                auxiliary.update(ordered_auxiliary)
            if self.pooling == "attentive_global":
                return global_summary, auxiliary
            if self.pooling == "ordered8":
                return ordered_summary, auxiliary
            combined = torch.cat(
                (
                    self.global_project(global_summary),
                    self.ordered_project(ordered_summary),
                ),
                dim=1,
            )
            return combined, auxiliary
        positions = torch.arange(hidden.shape[1], device=hidden.device).unsqueeze(0)
        mask = positions < lengths.unsqueeze(1)
        weight = mask.unsqueeze(-1).to(hidden.dtype)
        denominator = lengths.clamp_min(1).to(hidden.dtype).view(-1, 1)
        mean = (hidden * weight).sum(1) / denominator
        variance = ((hidden - mean.unsqueeze(1)).square() * weight).sum(1) / denominator
        return torch.cat((mean, variance.clamp_min(1e-7).sqrt()), dim=1), {}

    def forward(
        self, values: torch.Tensor, mask: torch.Tensor, return_auxiliary: bool = False
    ):
        hidden = self.encoder(
            input_values=values, attention_mask=mask
        ).last_hidden_state
        lengths = self.encoder._get_feat_extract_output_lengths(mask.sum(1)).to(
            hidden.device
        )
        aggregate, auxiliary = self.aggregate(hidden, lengths)
        shared = self.project(aggregate)
        output = self.base_head(shared), self.tone_head(shared)
        if return_auxiliary:
            return *output, auxiliary
        return output


def unfreeze_top_layers(encoder: nn.Module, count: int) -> list[str]:
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    layers = encoder.encoder.layers
    if not 1 <= count <= len(layers):
        raise ValueError(f"--unfreeze-layers must be from 1 to {len(layers)}")
    for layer in layers[-count:]:
        for parameter in layer.parameters():
            parameter.requires_grad_(True)
    return [
        f"encoder.layers.{index}" for index in range(len(layers) - count, len(layers))
    ]


def evaluate(model, loader, device, base_names, paths) -> tuple[dict, list[dict]]:
    model.eval()
    base_correct = tone_correct = joint_correct = tone_count = count = 0
    predictions = []
    attention_centers = []
    attention_entropies = []
    with torch.inference_mode():
        for values, mask, base, tone, rows in loader:
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                base_logits, tone_logits, auxiliary = model(
                    values.to(device), mask.to(device), return_auxiliary=True
                )
            base_pred, tone_pred = (
                base_logits.argmax(1).cpu(),
                tone_logits.argmax(1).cpu(),
            )
            valid = tone >= 0
            base_correct += (base_pred == base).sum().item()
            tone_correct += ((tone_pred == tone) & valid).sum().item()
            joint_correct += (
                ((base_pred == base) & (tone_pred == tone) & valid).sum().item()
            )
            count += base.numel()
            tone_count += valid.sum().item()
            centers = auxiliary.get("centers")
            entropies = auxiliary.get("entropy")
            if centers is not None:
                center_rows = centers.float().cpu().tolist()
                entropy_rows = entropies.float().cpu().tolist()
                attention_centers.extend(center_rows)
                attention_entropies.extend(entropy_rows)
            else:
                center_rows = [None] * rows.numel()
            predictions.extend(
                {
                    "path": paths[int(row)],
                    "base_true": base_names[int(b)],
                    "base_pred": base_names[int(bp)],
                    "tone_true": int(t) + 1 if int(t) >= 0 else "",
                    "tone_pred": int(tp) + 1,
                    "attention_centers": (
                        ";".join(f"{value:.5f}" for value in center_row)
                        if center_row is not None
                        else ""
                    ),
                }
                for row, b, bp, t, tp, center_row in zip(
                    rows, base, base_pred, tone, tone_pred, center_rows
                )
            )
    metrics = {
        "n": count,
        "base_accuracy": base_correct / count if count else None,
        "tone_n": tone_count,
        "tone_accuracy": tone_correct / tone_count if tone_count else None,
        "joint_accuracy": joint_correct / tone_count if tone_count else None,
    }
    if attention_centers:
        center_tensor = torch.tensor(attention_centers)
        entropy_tensor = torch.tensor(attention_entropies)
        metrics["attention_mean_centers"] = center_tensor.mean(0).tolist()
        metrics["attention_mean_entropy"] = entropy_tensor.mean(0).tolist()
    return metrics, predictions


def main() -> None:
    parser = argparse.ArgumentParser()
    default_config = Path("configs/unfrozen_classifier.json")
    parser.add_argument("--config", type=Path, default=default_config)
    parser.add_argument("--audio-cache", type=Path, default=Path("data/audio_16khz.pt"))
    parser.add_argument(
        "--endpoint-cache", type=Path, default=Path("data/audio_16khz_endpointed.pt")
    )
    parser.add_argument("--encoder", choices=sorted(MODEL_NAMES), required=True)
    parser.add_argument("--model-cache", type=Path, default=Path("models/huggingface"))
    parser.add_argument("--train-speaker", choices=["abe", "yue"], required=True)
    parser.add_argument(
        "--pooling",
        choices=["global", *TEMPORAL_POOLING, *sorted(ATTENTION_POOLING)],
        required=True,
    )
    parser.add_argument("--silence-augmentation", action="store_true")
    parser.add_argument("--maximum-silence-ms", type=int, default=500)
    parser.add_argument("--attention-diversity-weight", type=float, default=0.01)
    parser.add_argument("--attention-ordering-weight", type=float, default=0.01)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--unfreeze-layers", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--minimum-epochs", type=int, default=0)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--gradient-accumulation", type=int, default=4)
    parser.add_argument("--encoder-learning-rate", type=float, default=1e-5)
    parser.add_argument("--head-learning-rate", type=float, default=5e-4)
    parser.add_argument("--tone-loss-weight", type=float, default=1.0)
    parser.add_argument("--dropout", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=20260821)
    parser.add_argument("--device", default="cuda")
    apply_config_defaults(parser, requested_config_path(default_config))
    args = parser.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    device = torch.device(args.device)
    artifact = torch.load(args.audio_cache, map_location="cpu", weights_only=False)
    endpoint_metadata = None
    if args.silence_augmentation:
        endpoint_artifact = torch.load(
            args.endpoint_cache, map_location="cpu", weights_only=False
        )
        if endpoint_artifact["paths"] != artifact["paths"]:
            raise ValueError("Audio and endpoint caches contain different path orders")
        endpoint_metadata = endpoint_artifact["endpoint_metadata"]
        if endpoint_metadata is None:
            raise ValueError("Endpoint cache does not contain endpoint metadata")
    groups = [speaker_group(dataset) for dataset in artifact["datasets"]]
    base_names = sorted(set(artifact["bases"]))
    base_to_id = {name: i for i, name in enumerate(base_names)}
    base_targets = torch.tensor([base_to_id[name] for name in artifact["bases"]])
    tone_targets = torch.tensor(
        [
            int(tone) - 1 if tone in {"1", "2", "3", "4"} else -1
            for tone in artifact["tones"]
        ]
    )
    train, validation = split_training_speaker(
        artifact["paths"],
        artifact["bases"],
        groups,
        args.train_speaker,
        "stratified-base",
        0.15,
        args.seed,
    )
    external_speaker = "yue" if args.train_speaker == "abe" else "abe"
    external = [i for i, group in enumerate(groups) if group == external_speaker]
    oli = [i for i, group in enumerate(groups) if group == "oli"]
    indices = {
        "train": train,
        "validation": validation,
        "external": external,
        "oli": oli,
    }
    datasets = {
        name: AudioDataset(artifact, base_targets, tone_targets, rows)
        for name, rows in indices.items()
    }
    augmentation_config = SilenceAugmentationConfig(maximum_ms=args.maximum_silence_ms)
    loaders = {
        name: DataLoader(
            data,
            batch_size=args.batch_size,
            shuffle=name == "train",
            collate_fn=AudioCollator(
                augment=name == "train" and args.silence_augmentation,
                endpoint_metadata=endpoint_metadata,
                augmentation_config=augmentation_config,
            ),
            num_workers=0,
            pin_memory=True,
        )
        for name, data in datasets.items()
    }

    encoder = AutoModel.from_pretrained(
        MODEL_NAMES[args.encoder],
        revision=MODEL_REVISIONS[MODEL_NAMES[args.encoder]],
        cache_dir=args.model_cache,
        local_files_only=True,
    )
    unfrozen = unfreeze_top_layers(encoder, args.unfreeze_layers)
    model = FineTuneModel(encoder, args.pooling, len(base_names), args.dropout).to(
        device
    )
    encoder_parameters = [p for p in model.encoder.parameters() if p.requires_grad]
    head_parameters = [
        p for name, p in model.named_parameters() if not name.startswith("encoder.")
    ]
    trainable_names = {
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    }
    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_parameters, "lr": args.encoder_learning_rate},
            {"params": head_parameters, "lr": args.head_learning_rate},
        ],
        weight_decay=1e-2,
    )
    base_loss_fn, tone_loss_fn = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()

    best_scores = {
        "base_then_tone": (-1.0, -1.0),
        "tone": -1.0,
        "joint": -1.0,
    }
    best_states = {}
    best_epochs = {}
    stale = 0
    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        for step, (values, mask, base, tone, _) in enumerate(loaders["train"], 1):
            values, mask, base, tone = (
                values.to(device),
                mask.to(device),
                base.to(device),
                tone.to(device),
            )
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                base_logits, tone_logits, auxiliary = model(
                    values, mask, return_auxiliary=True
                )
                loss = base_loss_fn(base_logits, base)
                valid = tone >= 0
                if valid.any():
                    loss = loss + args.tone_loss_weight * tone_loss_fn(
                        tone_logits[valid], tone[valid]
                    )
                if "diversity_loss" in auxiliary:
                    loss = (
                        loss
                        + args.attention_diversity_weight * auxiliary["diversity_loss"]
                        + args.attention_ordering_weight * auxiliary["ordering_loss"]
                    )
                scaled_loss = loss / args.gradient_accumulation
            scaled_loss.backward()
            total_loss += loss.item() * base.numel()
            if step % args.gradient_accumulation == 0 or step == len(loaders["train"]):
                nn.utils.clip_grad_norm_([*encoder_parameters, *head_parameters], 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        validation_metrics, _ = evaluate(
            model, loaders["validation"], device, base_names, artifact["paths"]
        )
        selections = {
            "base_then_tone": (
                validation_metrics["base_accuracy"],
                validation_metrics["tone_accuracy"] or 0.0,
            ),
            "tone": validation_metrics["tone_accuracy"] or 0.0,
            "joint": validation_metrics["joint_accuracy"] or 0.0,
        }
        primary_selection = selections["base_then_tone"]
        record = {
            "epoch": epoch,
            "train_loss": total_loss / len(datasets["train"]),
            **validation_metrics,
        }
        history.append(record)
        print(json.dumps(record), flush=True)
        for criterion, selection in selections.items():
            if selection > best_scores[criterion]:
                best_scores[criterion] = selection
                best_epochs[criterion] = epoch
                best_states[criterion] = {
                    name: value.detach().cpu().clone()
                    for name, value in model.state_dict().items()
                    if name in trainable_names
                }
        if primary_selection > best_scores["base_then_tone"]:
            raise AssertionError("Primary checkpoint tracking is inconsistent")
        if best_epochs["base_then_tone"] == epoch:
            stale = 0
        else:
            if epoch >= args.minimum_epochs:
                stale += 1
            if epoch >= args.minimum_epochs and stale >= args.patience:
                break

    best_state = best_states["base_then_tone"]
    model.load_state_dict(best_state, strict=False)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    metrics = {
        "checkpoint_kind": "partial_finetune",
        "state_scope": "trainable_overlay",
        "encoder": args.encoder,
        "model_name": MODEL_NAMES[args.encoder],
        "model_revision": MODEL_REVISIONS[MODEL_NAMES[args.encoder]],
        "train_speaker": args.train_speaker,
        "external_speaker": external_speaker,
        "pooling": args.pooling,
        "unfreeze_layers": args.unfreeze_layers,
        "unfrozen_components": unfrozen,
        "seed": args.seed,
        "validation_strategy": "stratified-base",
        "base_vocabulary_size": len(base_names),
        "base_vocabulary": base_names,
        "architecture": {
            "dropout": args.dropout,
            "projection_size": 256,
            "temporal_bins": (
                TEMPORAL_POOLING[args.pooling]["bins"]
                if args.pooling in TEMPORAL_POOLING
                else 8
                if args.pooling in {"ordered8", "attentive_combined"}
                else None
            ),
            "frame_projection_size": (
                TEMPORAL_POOLING[args.pooling]["frame_projection_size"]
                if args.pooling in TEMPORAL_POOLING
                else 128
                if args.pooling in {"ordered8", "attentive_combined"}
                else None
            ),
            "attention_heads": (
                8 if args.pooling in {"ordered8", "attentive_combined"} else None
            ),
            "attention_size": 128 if args.pooling in ATTENTION_POOLING else None,
        },
        "training": {
            "epochs": args.epochs,
            "patience": args.patience,
            "minimum_epochs": args.minimum_epochs,
            "checkpoint_selection": "base_accuracy_then_tone_accuracy",
            "batch_size": args.batch_size,
            "gradient_accumulation": args.gradient_accumulation,
            "encoder_learning_rate": args.encoder_learning_rate,
            "head_learning_rate": args.head_learning_rate,
            "tone_loss_weight": args.tone_loss_weight,
            "attention_diversity_weight": args.attention_diversity_weight,
            "attention_ordering_weight": args.attention_ordering_weight,
            "silence_augmentation": args.silence_augmentation,
            "silence_augmentation_config": (
                asdict(augmentation_config) if args.silence_augmentation else None
            ),
        },
        "split_sizes": {name: len(data) for name, data in datasets.items()},
        "history": history,
        "selected_epochs": best_epochs,
        "selected_epoch": best_epochs["base_then_tone"],
    }
    predictions = {}
    for name in ("validation", "external", "oli"):
        metrics[name], predictions[name] = evaluate(
            model, loaders[name], device, base_names, artifact["paths"]
        )
    (args.output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    metadata, measured_metrics = split_run_record(metrics)
    checkpoint = create_checkpoint(
        state_dict=best_state,
        metadata=metadata,
        metrics=measured_metrics,
    )
    save_checkpoint(args.output_dir / "classifier.pt", checkpoint)
    for criterion, filename in (
        ("tone", "classifier_best_tone.pt"),
        ("joint", "classifier_best_joint.pt"),
    ):
        if best_epochs[criterion] == best_epochs["base_then_tone"]:
            alternate_metrics = measured_metrics
        else:
            model.load_state_dict(best_states[criterion], strict=False)
            alternate_metrics = {
                "history": history,
                **{
                    name: evaluate(
                        model, loaders[name], device, base_names, artifact["paths"]
                    )[0]
                    for name in ("validation", "external", "oli")
                },
            }
        alternate_metadata = copy.deepcopy(metadata)
        alternate_metadata["training"]["checkpoint_selection"] = criterion
        alternate_metadata["selected_epoch"] = best_epochs[criterion]
        alternate_checkpoint = create_checkpoint(
            state_dict=best_states[criterion],
            metadata=alternate_metadata,
            metrics=alternate_metrics,
        )
        save_checkpoint(args.output_dir / filename, alternate_checkpoint)
    with (args.output_dir / "predictions.tsv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        fields = [
            "split",
            "path",
            "base_true",
            "base_pred",
            "tone_true",
            "tone_pred",
            "attention_centers",
        ]
        writer = csv.DictWriter(
            handle, fieldnames=fields, delimiter="\t", lineterminator="\n"
        )
        writer.writeheader()
        for split, rows in predictions.items():
            for row in rows:
                writer.writerow({"split": split, **row})
    print(
        json.dumps(
            {name: metrics[name] for name in ("validation", "external", "oli")},
            indent=2,
        )
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/mandarin_finetune/predict_classifier.py
#!/usr/bin/env python3
"""Run one classifier checkpoint on an audio recording."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from checkpoint_utils import load_checkpoint
from extract_speech_features import decode_audio, pool_hidden
from train_syllable_classifier import Classifier
from train_unfrozen_classifier import FineTuneModel, unfreeze_top_layers
from transformers import AutoModel


def apply_trainable_overlay(
    model: torch.nn.Module, state_dict: dict[str, torch.Tensor]
) -> None:
    """Apply a partial state dictionary and reject unintended key differences."""
    expected_overlay = {
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    }
    provided = set(state_dict)
    missing_overlay = expected_overlay.difference(provided)
    unexpected_overlay = provided.difference(expected_overlay)
    if missing_overlay or unexpected_overlay:
        raise ValueError(
            "Invalid trainable overlay: "
            f"missing={sorted(missing_overlay)}, "
            f"unexpected={sorted(unexpected_overlay)}"
        )

    result = model.load_state_dict(state_dict, strict=False)
    if result.unexpected_keys:
        raise ValueError(f"Unexpected model keys: {result.unexpected_keys}")
    missing_trainable = expected_overlay.intersection(result.missing_keys)
    if missing_trainable:
        raise ValueError(f"Missing trainable model keys: {sorted(missing_trainable)}")


def load_encoder(metadata: dict, model_cache: Path, allow_download: bool):
    """Load the exact pretrained encoder revision named by checkpoint metadata."""
    return AutoModel.from_pretrained(
        metadata["model_name"],
        revision=metadata["model_revision"],
        cache_dir=model_cache,
        local_files_only=not allow_download,
    )


def predict_partial_finetune(
    checkpoint: dict,
    audio: torch.Tensor,
    device: torch.device,
    model_cache: Path,
    allow_download: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Reconstruct a partially fine-tuned model and return its logits."""
    metadata = checkpoint["metadata"]
    encoder = load_encoder(metadata, model_cache, allow_download)
    unfreeze_top_layers(encoder, metadata["unfreeze_layers"])
    model = FineTuneModel(
        encoder=encoder,
        pooling=metadata["pooling"],
        bases=len(metadata["base_vocabulary"]),
        dropout=metadata["architecture"]["dropout"],
    )
    apply_trainable_overlay(model, checkpoint["state_dict"])
    model.to(device).eval()

    values = audio.unsqueeze(0).to(device)
    mask = torch.ones_like(values, dtype=torch.long)
    with (
        torch.inference_mode(),
        torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == "cuda",
        ),
    ):
        return model(values, mask)


def predict_frozen_classifier(
    checkpoint: dict,
    audio: torch.Tensor,
    device: torch.device,
    model_cache: Path,
    allow_download: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Reconstruct a frozen encoder and its trained classifier heads."""
    metadata = checkpoint["metadata"]
    encoder = load_encoder(metadata, model_cache, allow_download).to(device).eval()
    values = audio.unsqueeze(0).to(device)
    mask = torch.ones_like(values, dtype=torch.long)
    with (
        torch.inference_mode(),
        torch.autocast(device_type=device.type, enabled=device.type == "cuda"),
    ):
        hidden = encoder(input_values=values, attention_mask=mask).last_hidden_state
    lengths = encoder._get_feat_extract_output_lengths(mask.sum(1)).cpu()
    global_features, temporal_features = pool_hidden(hidden.cpu(), lengths)
    features = global_features if metadata["pooling"] == "global" else temporal_features

    classifier = Classifier(
        shape=tuple(features.shape[1:]),
        pooling=metadata["pooling"],
        bases=len(metadata["base_vocabulary"]),
        dropout=metadata["architecture"]["dropout"],
    )
    classifier.load_state_dict(checkpoint["state_dict"], strict=True)
    classifier.to(device).eval()
    with torch.inference_mode():
        return classifier(features.to(device))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("checkpoint", type=Path)
    parser.add_argument("audio", type=Path)
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--model-cache", type=Path, default=Path("models/huggingface"))
    parser.add_argument("--ffmpeg", type=Path, default=Path("models/linux/ffmpeg"))
    parser.add_argument("--top-k", type=int, default=5)
    parser.add_argument("--allow-download", action="store_true")
    args = parser.parse_args()

    checkpoint = load_checkpoint(args.checkpoint)
    metadata = checkpoint["metadata"]
    device = torch.device(
        args.device if args.device != "cuda" or torch.cuda.is_available() else "cpu"
    )
    audio = decode_audio(args.audio, args.ffmpeg)
    predictors = {
        "partial_finetune": predict_partial_finetune,
        "frozen_encoder_classifier": predict_frozen_classifier,
    }
    predictor = predictors[metadata["checkpoint_kind"]]
    base_logits, tone_logits = predictor(
        checkpoint, audio, device, args.model_cache, args.allow_download
    )

    base_probabilities = base_logits.softmax(-1)[0].float().cpu()
    tone_probabilities = tone_logits.softmax(-1)[0].float().cpu()
    top_count = min(args.top_k, base_probabilities.numel())
    probabilities, indices = base_probabilities.topk(top_count)
    base_predictions = [
        {
            "label": metadata["base_vocabulary"][int(index)],
            "probability": float(probability),
        }
        for probability, index in zip(probabilities, indices)
    ]
    base = base_predictions[0]["label"]
    tone = int(tone_probabilities.argmax()) + 1
    result = {
        "checkpoint": str(args.checkpoint),
        "audio": str(args.audio),
        "model_name": metadata["model_name"],
        "model_revision": metadata["model_revision"],
        "base": base,
        "tone": tone,
        "joint": f"{base}{tone}",
        "base_top_k": base_predictions,
        "tone_probabilities": {
            str(index + 1): float(probability)
            for index, probability in enumerate(tone_probabilities)
        },
    }
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


## 4. Decode the release and calculate speech endpoints

Audio is held in CPU memory so 40 training epochs do not repeat
FLAC decoding. Endpoint detection does **not** trim training or
inference audio. It only identifies plausible leading and
trailing nonspeech for dynamic augmentation.

In [ ]:
sys.path.insert(0, str(WORK_DIR))
from speech_endpointing import EndpointConfig, detect_endpoints

SAMPLE_RATE = 16_000
metadata = pd.read_csv(DATASET_ROOT / "data/metadata.csv", keep_default_na=False)

def load_waveform(relative_name):
    path = DATASET_ROOT / "data" / relative_name
    waveform, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    waveform = waveform.mean(axis=1)
    if sample_rate != SAMPLE_RATE:
        divisor = math.gcd(int(sample_rate), SAMPLE_RATE)
        waveform = resample_poly(
            waveform,
            SAMPLE_RATE // divisor,
            int(sample_rate) // divisor,
        ).astype(np.float32)
    return torch.from_numpy(waveform.copy())

waveforms = []
endpoint_metadata = []
endpoint_config = EndpointConfig()
for index, row in metadata.iterrows():
    waveform = load_waveform(row.file_name)
    result = detect_endpoints(waveform, endpoint_config)
    waveforms.append(waveform)
    endpoint_metadata.append({
        "start_sample": result.start_sample,
        "end_sample": result.end_sample,
        "fallback": result.fallback,
        "fallback_reason": result.fallback_reason,
    })
    if (index + 1) % 500 == 0:
        print(f"Prepared {index + 1}/{len(metadata)}")

dataset_group = {
    "speaker_000000001": "tone_labeled",
    "speaker_000000002": "tone_labeled_yue",
    "speaker_000000003": "tone_unspecified",
}
artifact = {
    "format": 0,
    "waveforms": waveforms,
    "paths": metadata.recording_id.tolist(),
    "bases": metadata.base_syllable.tolist(),
    "tones": [str(value) for value in metadata.tone.tolist()],
    "datasets": [dataset_group[value] for value in metadata.speaker_id],
}
audio_cache = WORK_DIR / "audio_16khz.pt"
endpoint_cache = WORK_DIR / "audio_16khz_endpointed.pt"
torch.save(artifact, audio_cache)
torch.save({
    "format": 0,
    "paths": artifact["paths"],
    "endpoint_metadata": endpoint_metadata,
}, endpoint_cache)
print("Endpoint fallbacks:", sum(item["fallback"] for item in endpoint_metadata))
print(metadata.groupby("speaker_id").size())

## 5. Pin HuBERT and configure the reproduction

The model revision and hyperparameters match the successful
cluster run. The first download is cached in the Colab runtime.
`RUN_CONTROL_MODEL=True` optionally adds a second non-augmented
run, roughly doubling training time.

In [ ]:
MODEL_ID = "TencentGameMate/chinese-hubert-base"
MODEL_REVISION = "fce0375452b1dd6c080ac3248d423d4d037bc831"
MODEL_CACHE = WORK_DIR / "huggingface"
pretrained = AutoModel.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, cache_dir=MODEL_CACHE
)
del pretrained
torch.cuda.empty_cache()

CONFIG = {
    "attention_diversity_weight": 0.01,
    "attention_ordering_weight": 0.01,
    "batch_size": 8,
    "dropout": 0.2,
    "encoder_learning_rate": 1e-5,
    "epochs": 40,
    "gradient_accumulation": 4,
    "head_learning_rate": 5e-4,
    "maximum_silence_ms": 500,
    "minimum_epochs": 15,
    "patience": 6,
    "seed": 20260821,
    "tone_loss_weight": 1.0,
    "unfreeze_layers": 4,
}
config_path = WORK_DIR / "unfrozen_attention.json"
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
RUN_CONTROL_MODEL = False

## 6. Train the augmented ordered-attention model

Every epoch constructs new leading and trailing boundary
material. HuBERT frame representations therefore cannot be
cached: gradients update its top four layers during every batch.
The training program saves base-first, tone-first, and
joint-first validation checkpoints.

In [ ]:
def train_run(name, silence_augmentation):
    output_dir = WORK_DIR / "results" / name
    command = [
        sys.executable,
        str(WORK_DIR / "train_unfrozen_classifier.py"),
        "--config", str(config_path),
        "--audio-cache", str(audio_cache),
        "--endpoint-cache", str(endpoint_cache),
        "--encoder", "hubert",
        "--train-speaker", "abe",
        "--pooling", "ordered8",
        "--output-dir", str(output_dir),
        "--model-cache", str(MODEL_CACHE),
        "--device", "cuda",
    ]
    if silence_augmentation:
        command.append("--silence-augmentation")
    subprocess.run(command, cwd=WORK_DIR, check=True)
    return output_dir

augmented_output = train_run("ordered8_dynamic_silence", True)
control_output = (
    train_run("ordered8_no_augmentation", False)
    if RUN_CONTROL_MODEL else None
)

## 7. Review validation and external results

The primary checkpoint is selected by validation base accuracy,
with tone accuracy as its tie-breaker. Speaker-2 results are
reported only after selection.

In [ ]:
def show_metrics(output_dir):
    metrics = json.loads((output_dir / "metrics.json").read_text())
    rows = []
    for split in ("validation", "external", "oli"):
        result = metrics[split]
        rows.append({
            "split": split,
            "n": result["n"],
            "base_accuracy": result["base_accuracy"],
            "tone_accuracy": result["tone_accuracy"],
            "joint_accuracy": result["joint_accuracy"],
        })
    print("Selected epochs:", metrics["selected_epochs"])
    display(pd.DataFrame(rows).style.format({
        "base_accuracy": "{:.2%}",
        "tone_accuracy": lambda value: "" if value is None else f"{value:.2%}",
        "joint_accuracy": lambda value: "" if value is None else f"{value:.2%}",
    }))
    return metrics

augmented_metrics = show_metrics(augmented_output)
if control_output:
    control_metrics = show_metrics(control_output)

In [ ]:
predictions = pd.read_csv(augmented_output / "predictions.tsv", sep="\t")
external_tones = predictions[
    (predictions.split == "external") & (predictions.tone_true >= 0)
].copy()
external_tones["intended tone"] = external_tones.tone_true + 1
external_tones["predicted tone"] = external_tones.tone_pred + 1
matrix = pd.crosstab(
    external_tones["intended tone"],
    external_tones["predicted tone"],
    normalize="index",
).reindex(index=[1, 2, 3, 4], columns=[1, 2, 3, 4], fill_value=0)
figure, axis = plt.subplots(figsize=(5, 4))
image = axis.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
for row in range(4):
    for column in range(4):
        axis.text(column, row, f"{matrix.iloc[row, column]:.2f}", ha="center", va="center")
axis.set(
    xticks=range(4), yticks=range(4),
    xticklabels=[1, 2, 3, 4], yticklabels=[1, 2, 3, 4],
    xlabel="Predicted tone", ylabel="Intended tone",
    title="Speaker-2 normalized tone confusion",
)
figure.colorbar(image, ax=axis)
plt.show()

## 8. Reconstruct the selected checkpoint and inspect attention

The checkpoint contains only trainable layers. The frozen lower
HuBERT layers are restored from the pinned pretrained revision.

In [ ]:
from checkpoint_utils import load_checkpoint
from predict_classifier import apply_trainable_overlay, load_encoder
from train_unfrozen_classifier import FineTuneModel, unfreeze_top_layers

checkpoint = load_checkpoint(augmented_output / "classifier.pt")
checkpoint_metadata = checkpoint["metadata"]
encoder = load_encoder(checkpoint_metadata, MODEL_CACHE, allow_download=False)
unfreeze_top_layers(encoder, checkpoint_metadata["unfreeze_layers"])
model = FineTuneModel(
    encoder,
    checkpoint_metadata["pooling"],
    len(checkpoint_metadata["base_vocabulary"]),
    checkpoint_metadata["architecture"]["dropout"],
)
apply_trainable_overlay(model, checkpoint["state_dict"])
model.to(DEVICE).eval()

In [ ]:
@torch.inference_mode()
def predict_waveform(waveform):
    values = waveform.unsqueeze(0).to(DEVICE)
    mask = torch.ones_like(values, dtype=torch.long)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        base_logits, tone_logits, auxiliary = model(
            values, mask, return_auxiliary=True
        )
    return (
        base_logits.softmax(-1)[0].float().cpu(),
        tone_logits.softmax(-1)[0].float().cpu(),
        auxiliary["attention"][0].float().cpu(),
    )

def attention_plot(attention, title):
    positions = np.linspace(0, 1, attention.shape[1])
    figure, axis = plt.subplots(figsize=(10, 4))
    for head, weights in enumerate(attention.numpy(), 1):
        axis.plot(positions, weights, label=f"Head {head}")
    axis.set(xlabel="Relative time", ylabel="Attention weight", title=title)
    axis.legend(ncol=4, fontsize=8)
    figure.tight_layout()
    return figure

example = waveforms[int(metadata.index[metadata.speaker_id == "speaker_000000002"][0])]
base_probabilities, tone_probabilities, attention = predict_waveform(example)
print("Predicted tone:", int(tone_probabilities.argmax()) + 1)
display(attention_plot(attention, "Eight ordered attention heads"))

## 9. Try the fine-tuned model with Gradio

Record one isolated syllable. Colab creates a temporary public
Gradio URL; do not share it or use sensitive recordings. Inputs
remain in this runtime and are not added to the dataset.

In [ ]:
def prepare_audio(audio):
    if audio is None:
        raise gr.Error("Record or upload an isolated syllable.")
    sample_rate, waveform = audio
    waveform = np.asarray(waveform)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if np.issubdtype(waveform.dtype, np.integer):
        limit = max(abs(np.iinfo(waveform.dtype).min), np.iinfo(waveform.dtype).max)
        waveform = waveform.astype(np.float32) / limit
    else:
        waveform = waveform.astype(np.float32)
    if sample_rate != SAMPLE_RATE:
        divisor = math.gcd(int(sample_rate), SAMPLE_RATE)
        waveform = resample_poly(
            waveform, SAMPLE_RATE // divisor, int(sample_rate) // divisor
        ).astype(np.float32)
    return torch.from_numpy(waveform[: 10 * SAMPLE_RATE].copy())

def demo_predict(audio):
    waveform = prepare_audio(audio)
    base_probabilities, tone_probabilities, attention = predict_waveform(waveform)
    count = min(5, len(base_probabilities))
    values, indices = base_probabilities.topk(count)
    bases = {
        checkpoint_metadata["base_vocabulary"][int(index)]: float(value)
        for value, index in zip(values, indices)
    }
    tones = {
        f"Tone {index + 1}": float(value)
        for index, value in enumerate(tone_probabilities)
    }
    return bases, tones, attention_plot(attention, "Ordered attention")

with gr.Blocks() as demo:
    gr.Markdown(
        "# Fine-tuned Mandarin syllable and tone model\n"
        "Experimental predictions are not pronunciation assessments."
    )
    audio_input = gr.Audio(
        sources=["microphone", "upload"], type="numpy",
        label="Isolated syllable",
    )
    button = gr.Button("Classify", variant="primary")
    with gr.Row():
        base_output = gr.Label(label="Base syllable")
        tone_output = gr.Label(label="Tone")
    plot_output = gr.Plot(label="Attention")
    button.click(
        demo_predict, audio_input,
        [base_output, tone_output, plot_output],
    )
demo.launch(share=True, debug=False)

## Discussion

1. Did external tone accuracy fall near the 65–70% reference range?
2. Which tone confusions persisted across speakers?
3. Why can validation tone accuracy be high while external
   accuracy is substantially lower?
4. How does dynamic augmentation differ from caching one
   augmented HuBERT representation?
5. What additional annotations would be required before using
   these outputs for pronunciation assessment?